In [1]:
import os
import time
import sys
import json
import numpy as np
import time

import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams["lines.linewidth"] = 2.0

import seaborn as sns
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision.datasets import STL10
from torchvision import transforms
import torch.utils.data as data
from torch.utils.data import DataLoader, Dataset
from torch.utils.data import random_split

!pip install pytorch_lightning as pl


ERROR: Could not find a version that satisfies the requirement as (from versions: none)
ERROR: No matching distribution found for as


In [2]:
!pip  install --quiet pytorch_lightning

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 31.6 MB/s eta 0:00:00


In [3]:
import pytorch_lightning as pl

In [4]:
from pytorch_lightning.callbacks import ModelCheckpoint, LearningRateMonitor

data_path = "data_path"
checkpoint_path = "checkpoint_path"

def set_seed(seed):
  np.random.seed(seed)
  torch.manual_seed(seed)
  if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed(42)

torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False

device = torch.device("cuda:0")if torch.cuda.is_available()else torch.device("cpu")
print("using this device", device)

using this device cuda:0


In [5]:
import urllib.request
from urllib.error import HTTPError
base_url = "https://raw.githubusercontent.com/phlippe/saved_models/main/tutorial17/"
pretrained_filename = ["SimCLR.ckpt", "ResNet.ckpt",
                    "tensorboards/SimCLR/events.out.tfevents.SimCLR",
                    "tensorboards/classification/ResNet/events.out.tfevents.ResNet"]
pretrained_filename += [f"LogisticRegression_{size}.ckpt" for size in [10, 20, 50, 100, 200, 500]]
os.makedirs(checkpoint_path, exist_ok = True)

def pretrained_file(base_url:str, pretrained_filename:str):
  for file_name in pretrained_filename:
    file_path = os.path.join(checkpoint_path, file_name)
    if "/" in file_name:
      os.makedirs(os.path.dirname(file_path), exist_ok = True)
    if not os.path.isfile(file_path):
      file_url = base_url + file_name
      print("downloading this file_url", {file_path})
      try:
        urllib.request.urlretrieve(file_url, file_path)
      except HTTPError as e:
        print(f"downloading this filename from google_drive\n", e)


if __name__ == "__main__":
  pretrained_file(base_url, pretrained_filename)

downloading this file_url {'checkpoint_path/SimCLR.ckpt'}
downloading this file_url {'checkpoint_path/ResNet.ckpt'}
downloading this file_url {'checkpoint_path/tensorboards/SimCLR/events.out.tfevents.SimCLR'}
downloading this file_url {'checkpoint_path/tensorboards/classification/ResNet/events.out.tfevents.ResNet'}
downloading this file_url {'checkpoint_path/LogisticRegression_10.ckpt'}
downloading this file_url {'checkpoint_path/LogisticRegression_20.ckpt'}
downloading this file_url {'checkpoint_path/LogisticRegression_50.ckpt'}
downloading this file_url {'checkpoint_path/LogisticRegression_100.ckpt'}
downloading this file_url {'checkpoint_path/LogisticRegression_200.ckpt'}
downloading this file_url {'checkpoint_path/LogisticRegression_500.ckpt'}


In [6]:
class constrastive_transforms(object):
  def __init__(self, base_transforms, n_views = 2):
    super().__init__()
    self.base_transforms = base_transforms
    self.n_views = n_views

  def __call__(self, x):
    return [self.base_transforms(x) for i in range(self.n_views)]

In [7]:
contrastive_transform = transforms.Compose([transforms.ToTensor(),
                                            transforms.RandomResizedCrop((32, 32), scale = (0.8, 1.0), ratio = (0.8, 1.0)),
                                            transforms.RandomHorizontalFlip(p = 0.5),
                                            transforms.ColorJitter(brightness = 0.5, contrast = 0.5,
                                                                          hue = 0.5,
                                                                          saturation = 0.5),
                                            transforms.RandomRotation(degrees = 0.8),
                                            transforms.RandomGrayscale(p = 0.5),
                                            transforms.GaussianBlur(kernel_size = 7),
                                            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

In [8]:
train_dataset = STL10(root = data_path, download = True, transform = constrastive_transforms(contrastive_transform,n_views = 2))
test_dataset = STL10(root = data_path,  download = True, transform = constrastive_transforms(contrastive_transform, n_views = 2))
print("train_dataset", train_dataset)
print("test_dataset", test_dataset)

100%|██████████| 2.64G/2.64G [03:07<00:00, 14.1MB/s]


train_dataset Dataset STL10
    Number of datapoints: 5000
    Root location: data_path
    Split: train
    StandardTransform
Transform: <__main__.constrastive_transforms object at 0x7a583b313cb0>
test_dataset Dataset STL10
    Number of datapoints: 5000
    Root location: data_path
    Split: train
    StandardTransform
Transform: <__main__.constrastive_transforms object at 0x7a583b2c6d50>


In [9]:
class SIMCLR(pl.LightningModule):
  def __init__(self, hidden_dim, lr, temperature, weight_decay,  max_epochs = 500):
    super().__init__()
    self.save_hyperparameters()
    self.hidden_dim = hidden_dim
    assert self.hparams.temperature > 0, "the temperature must be a positive interger"

    # Fix: Replace 'pretrained=False' with 'weights=None'
    self.convet = torchvision.models.resnet18(weights = None, num_classes = 4 * hidden_dim)
    self.convet.fc = nn.Sequential(self.convet.fc,
                                   nn.ReLU(inplace = True),
                                   nn.Linear(4*hidden_dim, hidden_dim))

  def configure_optimizers(self):
    optimizer = torch.optim.Adam(self.parameters(), lr = self.hparams.lr, weight_decay = self.hparams.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max = self.hparams.max_epochs, eta_min = self.hparams.lr/50)

    return [optimizer], [scheduler]

  def calculate_loss(self, batch, mode = "train"):
    img, _ = batch
    img = torch.cat(img, dim = 0)
    feature = self.convet(img)
    cos_sim = F.cosine_similarity(feature[:,None,:],feature[None,:,:], dim = -1)
    self_mask = torch.eye(cos_sim.shape[0], dtype = torch.bool, device = cos_sim.device)
    cos_sim.masked_fill_(self_mask, -8e15)
    sim_pos = self_mask.roll(shifts = cos_sim.shape[0]//2, dims = 0)
    cos_sim = cos_sim / self.hparams.temperature
    nll = -cos_sim[sim_pos] + torch.logsumexp(cos_sim, dim = -1)
    # Fix: Return the scalar mean of nll
    return nll.mean()





  def training_step(self, batch, batch_idx):
    return self.calculate_loss(batch, mode = "train")

  def validation_step(self, batch, batch_idx):
    return self.calculate_loss(batch, mode = "val")

  def test_step(self, batch, batch_idx):
    return self.calculate_loss(batch, mode = "test")

In [10]:
def training_sclr(batch_size, max_epochs = 100, **kwargs):
  root_dir = os.path.join(checkpoint_path, "SIMCLR")
  trainer = pl.Trainer(default_root_dir = root_dir,
                       max_epochs = max_epochs, min_epochs = 20,
                       devices = 1,
                       accelerator = "auto",
                       callbacks =[ModelCheckpoint(save_weights_only = True, mode = "max"),LearningRateMonitor("epoch")])
  trainer.logger._default_hp_metrics = False
  trainer.logger._log_graph = True

  pretrained_filename = os.path.join(checkpoint_path, "SIMCLR.ckpt")
  if os.path.isfile(pretrained_filename):
    model = SIMCLR.load_from_checkpoint_path(pretrained_filename)
  else:
    pl.seed_everything(42)
    train_loader = data.DataLoader(train_dataset, batch_size = batch_size,shuffle = True, num_workers = 0, pin_memory = True)
    val_loader = data.DataLoader(test_dataset, batch_size = batch_size,  shuffle = False, num_workers = 0, pin_memory = True)
    model = SIMCLR(max_epochs = max_epochs, **kwargs)
    trainer.fit(model, train_loader, val_loader)
    model = SIMCLR.load_from_checkpoint(trainer.checkpoint_callback.best_model_path)

  trainer_results = trainer.test(model, train_loader, verbose = False)
  val_results  = trainer.validate(model, val_loader, verbose = False)

  return model

In [11]:
simclr_model = training_sclr(hidden_dim = 128,
                            lr = 5e-4,
                            temperature = 0.07,
                            max_epochs = 100,
                            batch_size = 256,
                            weight_decay = 0.0)

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning_fabric.utilities.seed:Seed set to 42
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name   ┃ Type   ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ convet │ ResNet │ 11.5 M │ train │     0 │
└───┴────────┴────────┴────────┴───────┴───────┘

Trainable params: 11.5 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 11.5 M                                                                                               
Total estimated model params size (MB): 46.019                                                                     
Modules in train mode: 71                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.13/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

...

INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

/usr/local/lib/python3.13/dist-packages/pytorch_lightning/trainer/connectors/data_connector.py:485: Your `test_dataloader`'s sampler has shuffling enabled, it is strongly recommended that you turn shuffling off for val/test dataloaders.


INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

In [12]:
class LogisticRegression(pl.LightningModule):
  def __init__(self, hidden_dim,  lr,  weight_decay, num_classes, max_epochs = 100):
    super().__init__()
    self.save_hyperparameters()
    self.linear = nn.Linear(hidden_dim, num_classes)

  def configure_optimizers(self):
    optimizer = torch.optim.AdamW(self.parameters(), lr = self.hparams.lr, weight_decay = self.hparams.weight_decay)
    scheduler = torch.lr_scheduler.MultiStepLR(optimizer, milestones = [int(self.hparams.max_epochs * 0.6),
                                                                        int(self.hparams.max_epochs*0.4)], gamma = 0.1)
    return [optimizer], [scheduler]


  def _calculate_loss(self, batch, mode = "train"):
    img, labels = batch
    pred = self.linear(img)
    loss = F.cross_entropy(pred, labels)
    acc = (pred.argmax(dim = -1) == labels).float().mean()

    self.log(f"{mode}_loss" + loss)
    self.log(f"{mode}_acc" + acc)

    return loss, acc


  def training_step(self, batch, batch_idx):
    return self._calculate_loss(batch, mode = "train")


  def validation_step(self, batch, batch_idx):
    return self._calculate_loss(batch, mode = "val")

  def test_step(self, batch, batch_idx):
    return self._calculate_loss(batch, mode = "test")

In [13]:
train_transforms = transforms.Compose([transforms.ToTensor(),
                                       transforms.RandomHorizontalFlip(p = 0.5),
                                       transforms.RandomResizedCrop((32,32), scale = (0.8, 1.0), ratio = (0.8, 1.0)),transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])
test_transforms = transforms.Compose([transforms.ToTensor(),
                                      transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

train_dataset = STL10(root = data_path, split = "train",download = True, transform = train_transforms)
test_dataset = STL10(root = data_path, split = "test",download = True, transform = test_transforms)

In [14]:
from copy import deepcopy
from tqdm.notebook import tqdm

In [15]:
@torch.no_grad()
def prepare_data(model, dataset):
  network  = deepcopy(model.convet)
  network.fc = nn.Identity()
  network.eval()
  network.to(device)


  data_loader = data.DataLoader(dataset,  batch_size = 256, shuffle = True, num_workers = 0, pin_memory = True)
  feats, labels = [], []

  for batch_label, batch_target in (data_loader):
    batch_feats = network(batch_label.to(device))
    feats.append(batch_feats.detach().cpu())
    labels.append(batch_target.detach().cpu())

  feats = torch.cat(feats, dim = 0)
  labels = torch.cat(labels, dim = 0)


  labels, idx = labels.sort()
  feats = feats[idx]


  return data.TensorDataset(feats, labels)

In [16]:
train_datasets = prepare_data(simclr_model, train_dataset)
test_datasets = prepare_data(simclr_model, test_dataset)

print("using train_dataset", train_datasets)
print("using the test_dataset", test_datasets)

using train_dataset <torch.utils.data.dataset.TensorDataset object at 0x7a583b313cb0>
using the test_dataset <torch.utils.data.dataset.TensorDataset object at 0x7a579ce6ae90>


In [17]:
def trainer_logistic(batch_size, train_datasets, test_datasets, model_name,max_epochs = 100, **kwargs):
  root_dir = os.path.join(checkpoint_path, "LogisticRegression")

  trainer = pl.Trainer(default_root_dir = os.path.join(root_dir),
                       devices = 1,
                       accelerator = "auto",
                       max_epochs = max_epochs,
                       min_epochs = 10,
                       callbacks = [ModelCheckpoint(save_weights_only = True, mode = "max"), LearningRateMonitor("epoch")])
  trainer.logger._default_hp_metrics = False
  trainer.logger._log_graph = True

  train_dataloader = data.DataLoader(train_datasets, batch_size = batch_size, shuffle = True, num_workers = 0, pin_memory = True)
  test_dataloader = data.DataLoader(test_datasets, batch_size = batch_size, shuffle = False, num_workers = 0, pin_memory = True)

  pretrained_filename = os.path.join(checkpoint_path, f"LogisticRegression{model_name}.ckpt")
  if os.path.isfile(pretrained_filename):
    model = LogisticRegression.load_from_checkpoint(pretrained_filename)
  else:
    pl.seed_everything(42)
    model = LogisticRegression(max_epochs = max_epochs, **kwargs)
    trainer.fit(model,train_dataloader, test_dataloader)

  trainer_results = trainer.test(model, train_dataloader, verbose = False)
  test_results = trainer.test(model, test_dataloader, verbose = False)

  results = {"test_acc":trainer_results[0]["test_acc"],
             "test_acc":test_results[0]["test_acc"]}

  return model

In [18]:
class ResNet(pl.LightningModule):
  def __init__(self,  lr, weight_decay, num_classes, max_epochs = 100):
    super().__init__()
    self.save_hyperparameters()
    self.linear = torchvision.models.resnet18(num_classes = num_classes)


  def configure_optimizers(self):
    optimizer = torch.optim.AdamW(self.parameters(), lr = self.hparams.lr, weight_decay = self.hparams.weight_decay)
    scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones = [int(self.hparams.max_epochs * 0.6), int(self.hparams.max_epochs * 0.4)], gamma = 0.1)

    return [optimizer], [scheduler]


  def _calculate_loss(self, batch, mode = "train"):
    label, target = batch
    pred = self.linear(label)
    loss = F.cross_entropy(pred, target)
    acc = (pred.argmax(dim = -1) == target).float().mean()

    self.log(f"{mode}_loss", loss)
    self.log(f"{mode}_acc", acc)
    return loss


  def training_step(self, batch, batch_idx):
    return self._calculate_loss(batch, mode = "train")

  def validation_step(self, batch, batch_idx):
    return self._calculate_loss(batch, mode = "val")

  def test_step(self, batch, batch_idx):
    return self._calculate_loss(batch, mode = "test")

In [19]:
train_transforms_resnet = transforms.Compose([transforms.ToTensor(),
                                       transforms.GaussianBlur(kernel_size = 5),
                                       transforms.RandomResizedCrop((32, 32), scale = (0.8, 1.0), ratio = (0.8, 1.0)), transforms.RandomHorizontalFlip(p = 0.5),
                                       transforms.ColorJitter(brightness = 0.5, contrast = 0.5, saturation = 0.5, hue = 0.5),
                                       transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])
train_datasets = STL10(root = data_path, split = "train", download = True, transform = train_transforms_resnet)
test_datasets = STL10(root = data_path, split = "test", download = True, transform = test_transforms)

In [20]:
def resnet_trainer(batch_size, model_name, max_epochs = 100, **kwargs):
  root_dir = os.path.join(checkpoint_path, "ResNet_simclr")
  trainer = pl.Trainer(default_root_dir = root_dir,
                       max_epochs = max_epochs,
                       min_epochs = 10,
                       accelerator = "auto",
                       devices = 1,
                       callbacks = [ModelCheckpoint(save_weights_only = True, mode = "max"), LearningRateMonitor("epoch")])
  trainer.logger._log_graph = True
  trainer.logger._default_hp_metrics = False

  pretrained_filename = os.path.join(checkpoint_path, f"ResNet_simclr{model_name}.ckpt")
  if os.path.isfile(pretrained_filename):
    model = ResNet.load_from_checkpoint(pretrained_filename)
  else:
    pl.seed_everything(42)
    model = ResNet(max_epochs = max_epochs, **kwargs)
    trainer_loader = data.DataLoader(train_datasets, batch_size = 256, shuffle = True, pin_memory = True, num_workers = 0)
    test_dataloader = data.DataLoader(test_datasets, batch_size = 256, shuffle = False, pin_memory = True, num_workers = 0) # Renamed for clarity
    trainer.fit(model, trainer_loader, test_dataloader) # Pass test_dataloader as val_loader during fit
    model = ResNet.load_from_checkpoint(trainer.checkpoint_callback.best_model_path)

  # Evaluate on the training set
  train_evaluation_results = trainer.test(model, trainer_loader, verbose = False)
  # Evaluate on the dedicated test set
  test_evaluation_results = trainer.test(model, test_dataloader, verbose = False) # Use trainer.test for test accuracy

  results = {"train":train_evaluation_results[0]["test_acc"],
             "test":test_evaluation_results[0]["test_acc"]} # Now retrieves test_acc

  return model, results

In [21]:
resnet_model, resnet_results = resnet_trainer(model_name = "Resnet",
                                              batch_size = 64,
                                              max_epochs = 100,
                                              lr = 1e-4,
                                              weight_decay = 2e-5,
                                              num_classes = 10
                                              )
print(f"the accuracy of the train_results is {100 * resnet_results['train']}")
print(f"the accuracy of the test_results is {100 * resnet_results['test']}")

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning_fabric.utilities.seed:Seed set to 42
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name   ┃ Type   ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ linear │ ResNet │ 11.2 M │ train │     0 │
└───┴────────┴────────┴────────┴───────┴───────┘

Trainable params: 11.2 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 11.2 M                                                                                               
Total estimated model params size (MB): 44.727                                                                     
Modules in train mode: 68                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=100` reached.


INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

the accuracy of the train_results is 70.03999948501587
the accuracy of the test_results is 10.487499833106995
